# 7.6 Training Loops, Validation, and Experiment Hygiene

[Back to neural networks index](../7_neural_networks.ipynb)

This notebook turns isolated train steps into a disciplined experiment loop with validation, reproducibility, and basic checkpointing.


## 7.6.1 Train mode versus eval mode

### Why It Matters
Some layers behave differently during training and evaluation. Dropout is the cleanest way to see why `model.train()` and `model.eval()` matter.


In [ ]:
import torch
from torch import nn

torch.manual_seed(19)
model = nn.Sequential(nn.Linear(4, 4), nn.Dropout(p=0.5), nn.Linear(4, 1))
x = torch.ones(3, 4)

model.train()
train_out = model(x)
model.eval()
eval_out = model(x)
{
    "train_outputs": train_out.detach().flatten().round(decimals=3).tolist(),
    "eval_outputs": eval_out.detach().flatten().round(decimals=3).tolist(),
}


### Interpretation
The same model can produce different outputs depending on its mode. That difference is intentional, not a bug.


## 7.6.2 Epochs and mini-batches

### Why It Matters
An epoch means one full pass through the dataset, usually as many mini-batches. Logging loss per epoch makes training progress easier to read.


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(20)
X = torch.randn(96, 3)
y = ((X[:, 0] - X[:, 1]) > 0).float().unsqueeze(1)
loader = DataLoader(TensorDataset(X, y), batch_size=16, shuffle=True)
model = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())
opt = torch.optim.Adam(model.parameters(), lr=0.03)
loss_fn = nn.BCELoss()
epoch_losses = []
for _ in range(4):
    running = 0.0
    for batch_X, batch_y in loader:
        opt.zero_grad()
        loss = loss_fn(model(batch_X), batch_y)
        loss.backward()
        opt.step()
        running += loss.item()
    epoch_losses.append(round(running / len(loader), 4))
epoch_losses


### Interpretation
Loss is noisy at the batch level, so the epoch average is often the easier learning signal to monitor.


## 7.6.3 Validation tracking

### Why It Matters
A training loss can keep falling even while validation quality stops improving. That is why model selection should watch held-out data.


In [ ]:
import torch
from torch import nn
from sklearn.model_selection import train_test_split

torch.manual_seed(21)
X = torch.randn(220, 6)
y = ((X[:, 0] + 0.5 * X[:, 1] - X[:, 2]) > 0).float().unsqueeze(1)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=21)
model = nn.Sequential(nn.Linear(6, 20), nn.ReLU(), nn.Linear(20, 1), nn.Sigmoid())
opt = torch.optim.Adam(model.parameters(), lr=0.03)
loss_fn = nn.BCELoss()
history = []
for _ in range(10):
    opt.zero_grad()
    train_loss = loss_fn(model(X_train), y_train)
    train_loss.backward()
    opt.step()
    with torch.no_grad():
        val_loss = loss_fn(model(X_val), y_val)
    history.append((round(float(train_loss.item()), 4), round(float(val_loss.item()), 4)))
history[:5]


### Interpretation
Training and validation curves tell you whether the model is learning, stalling, or beginning to overfit.


## 7.6.4 Checkpointing basics

### Why It Matters
A checkpoint stores learned parameters so the training state can be reused later. This example saves the best-so-far state in memory and reloads it.


In [ ]:
import copy
import torch
from torch import nn

torch.manual_seed(22)
X = torch.randn(80, 4)
y = ((X.sum(dim=1) > 0).float().unsqueeze(1))
model = nn.Sequential(nn.Linear(4, 10), nn.ReLU(), nn.Linear(10, 1), nn.Sigmoid())
opt = torch.optim.Adam(model.parameters(), lr=0.03)
loss_fn = nn.BCELoss()

best_state = None
best_loss = float('inf')
for _ in range(8):
    opt.zero_grad()
    loss = loss_fn(model(X), y)
    loss.backward()
    opt.step()
    if loss.item() < best_loss:
        best_loss = loss.item()
        best_state = copy.deepcopy(model.state_dict())

restored = nn.Sequential(nn.Linear(4, 10), nn.ReLU(), nn.Linear(10, 1), nn.Sigmoid())
restored.load_state_dict(best_state)
{"best_loss": round(float(best_loss), 4), "restored_parameter_tensors": len(restored.state_dict())}


### Interpretation
Checkpointing is how you preserve the best version of the model rather than only the last version.


## 7.6.5 Seed control and reproducibility

### Why It Matters
Random initialization and shuffled batches mean repeated training runs can differ. Controlling seeds helps you separate modeling changes from randomness.


In [ ]:
import torch
from torch import nn

def train_once(seed):
    torch.manual_seed(seed)
    X = torch.randn(60, 3)
    y = ((X[:, 0] - X[:, 2]) > 0).float().unsqueeze(1)
    model = nn.Sequential(nn.Linear(3, 6), nn.ReLU(), nn.Linear(6, 1), nn.Sigmoid())
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = nn.BCELoss()
    for _ in range(50):
        opt.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        opt.step()
    return round(float(loss.item()), 4)

{"seed_1_run_a": train_once(1), "seed_1_run_b": train_once(1), "seed_2_run": train_once(2)}


### Interpretation
Matching seeded runs make experiments easier to debug and compare honestly.


## 7.6.6 A clean full training loop

### Why It Matters
This final subsection combines mode switching, batching, validation, and history logging into one compact loop that you can reuse later.


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

torch.manual_seed(23)
X = torch.randn(180, 5)
y = ((X[:, 0] + X[:, 1] - 0.5 * X[:, 4]) > 0).float().unsqueeze(1)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=23)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=24, shuffle=True)

model = nn.Sequential(nn.Linear(5, 12), nn.ReLU(), nn.Linear(12, 1), nn.Sigmoid())
opt = torch.optim.Adam(model.parameters(), lr=0.03)
loss_fn = nn.BCELoss()
history = []
for _ in range(5):
    model.train()
    for batch_X, batch_y in train_loader:
        opt.zero_grad()
        loss = loss_fn(model(batch_X), batch_y)
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(X_val), y_val).item()
    history.append(round(float(val_loss), 4))
history


### Interpretation
The loop is still short, but it now reflects the structure of a real experiment instead of a one-off toy cell.
